In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("lc_accepted_clean.csv", low_memory=False)

print("Rows:", df.shape[0])
print("Default rate:", df["default"].mean())

Rows: 1345350
Default rate: 0.19964990522912254


In [3]:
df.dtypes

loan_amnt         float64
term               object
int_rate          float64
installment       float64
purpose            object
annual_inc        float64
emp_length         object
home_ownership     object
dti               float64
delinq_2yrs       float64
inq_last_6mths    float64
open_acc          float64
pub_rec           float64
revol_bal         float64
revol_util        float64
total_acc         float64
issue_d            object
default             int64
dtype: object

I need to clean term and emp_length columns. They are currently strings but should be numeric.

In [4]:
df["term"].value_counts()

 36 months    1020768
 60 months     324582
Name: term, dtype: int64

In [5]:
df["term"] = df["term"].str.extract("(\d+)").astype(int)

df["term"].value_counts()

36    1020768
60     324582
Name: term, dtype: int64

In [6]:
df["emp_length"].value_counts()

10+ years    442209
2 years      121751
< 1 year     108065
3 years      107602
1 year        88495
5 years       84154
4 years       80558
6 years       62735
8 years       60704
7 years       59624
9 years       50937
Name: emp_length, dtype: int64

In [7]:
def clean_emp_length(x):
    if pd.isna(x):
        return np.nan
    if "<" in x:
        return 0
    if "10+" in x:
        return 10
    return int(x.split()[0])

df["emp_length"] = df["emp_length"].apply(clean_emp_length)

In [8]:
df["emp_length"].describe()

count    1.266834e+06
mean     5.965847e+00
std      3.691171e+00
min      0.000000e+00
25%      2.000000e+00
50%      6.000000e+00
75%      1.000000e+01
max      1.000000e+01
Name: emp_length, dtype: float64

I'll work with the natural log of income (+1 inside the log, to avoid log(0)), since it has a skewed distribution.

In [9]:
df["log_annual_inc"] = np.log1p(df["annual_inc"])
df = df.drop(columns=["annual_inc"])

I convert the issue variables to dates, in order to consider the dynamics in the train-test split

In [10]:
df["issue_d"] = pd.to_datetime(df["issue_d"])
df["issue_year"] = df["issue_d"].dt.year

In [11]:
train = df[df["issue_year"] <= 2016].copy()
test  = df[df["issue_year"] >= 2017].copy()

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("Train default rate:", train["default"].mean())
print("Test default rate:", test["default"].mean())

Train shape: (1119711, 19)
Test shape: (225639, 19)
Train default rate: 0.1969758267981649
Test default rate: 0.21291975234777677


## Feature Set and Model Split

In [12]:
target = "default"

drop_cols = ["default", "issue_d", "issue_year"]

X_train = train.drop(columns=drop_cols)
y_train = train[target]

X_test  = test.drop(columns=drop_cols)
y_test  = test[target]

In [13]:
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)